# 无插值敏感性分析 — 眼动 LMM（Pairwise Deletion）

**目的**：审稿意见 #6.2 — 验证线性插值（27%缺失率）未人为制造虚假趋势。

**方法**：跳过插值步骤，将 0 值视为缺失（与 EYE/2 一致），使用 pairwise deletion 重跑三个 LMM，
与原始线性插值结果并列对比。

**输入**：`眼动数据预处理文件.xlsx`（EYE/1 输出，插值前的数据）

**对比基准**：`code/EYE/5.混合线性模型.ipynb` 的线性插值 LMM 结果

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import statsmodels.formula.api as smf
import statsmodels.api as sm
import os

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

In [ ]:
# 读取插值前的原始数据
input_file = "眼动数据预处理文件.xlsx"
if not os.path.exists(input_file):
    input_file = r"../数据文件/EYE/眼动数据预处理文件.xlsx"

print(f"读取: {input_file}")
df = pd.read_excel(input_file)
print(f"原始记录数: {len(df)}")
print(f"列名: {list(df.columns)}")
df.head()

In [ ]:
# 列名映射和清理
# EYE/1 输出的列: 组别(A/B), 性别, 被试者, 天数, 阶段, AOI转换次数, 静态注视熵(SGE), 眼跳注视熵(GTE)
# EYE/5 的列: 阶段, AOI转换次数, 静态注释熵(SGE), 眼跳注视熵(GTE), 组别(Alcohol/Control), 性别, 受试者, 飞行天数

# 统一列名
df = df.rename(columns={
    '被试者': '受试者',
    '天数': '飞行天数'
})

# 统一组别标签
df['组别'] = df['组别'].replace({'A': 'Alcohol', 'B': 'Control'})

# 统一阶段名称
phase_map = {
    '起飞阶段': '起飞',
    '第1次转弯': '转弯',
    '第2次转弯': '转弯',
    '第3次转弯': '转弯',
    '第4次转弯': '转弯',
    '巡航阶段': '巡航',
    '降落阶段': '降落'
}
df['阶段'] = df['阶段'].replace(phase_map)

print(f"组别: {df['组别'].unique()}")
print(f"阶段: {df['阶段'].unique()}")
print(f"飞行天数: {sorted(df['飞行天数'].unique())}")
print(f"受试者数: {df['受试者'].nunique()}")

In [ ]:
# 将 0 值替换为 NaN（与 EYE/2 插值脚本一致的处理）
metrics = ['AOI转换次数', '静态注视熵(SGE)', '眼跳注视熵(GTE)']

print("=== 替换前缺失统计 ===")
for m in metrics:
    zero_count = (df[m] == 0).sum()
    nan_count = df[m].isna().sum()
    total = len(df)
    print(f"{m}: 0值={zero_count} ({zero_count/total*100:.1f}%), NaN={nan_count} ({nan_count/total*100:.1f}%)")

# 替换
for m in metrics:
    df[m] = df[m].replace(0, np.nan)

print(f"\n替换后总缺失: {df[metrics].isna().sum().sum()} / {len(df) * 3}")
print(f"总体缺失率: {df[metrics].isna().sum().sum() / (len(df) * 3) * 100:.2f}%")

In [ ]:
# 保存处理后的数据（便于检查）
df.to_excel("眼动数据_无插值_格式对齐.xlsx", index=False)

# 统计各模型的可用样本量
print("=== Pairwise Deletion 后的样本量 ===")
for m in metrics:
    n_complete = df[m].notna().sum()
    n_total = len(df)
    print(f"{m}: {n_complete}/{n_total} ({n_complete/n_total*100:.1f}%)")

In [ ]:
# ========================================
# 混合线性模型（无插值，pairwise deletion）
# 公式与 EYE/5 完全一致
# ========================================

def extract_result(result, model_name):
    """提取 LMM 结果为 DataFrame"""
    summary_df = pd.DataFrame({
        "coef": result.params,
        "std_err": result.bse,
        "z_value": result.tvalues,
        "p_value": result.pvalues
    })
    summary_df["显著性"] = summary_df["p_value"].apply(
        lambda p: "显著" if p < 0.05 else "不显著"
    )
    summary_df["模型"] = model_name
    return summary_df

In [ ]:
# 模型 1: AOI转换次数
print("=" * 70)
print("模型 1: AOI转换次数 ~ 组别 * 阶段 * 飞行天数 (无插值)")
print("=" * 70)

model_AOI_raw = smf.mixedlm(
    "AOI转换次数 ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_AOI_raw = model_AOI_raw.fit(method="powell", maxiter=500)
print(result_AOI_raw.summary())

In [ ]:
# 模型 2: 静态注视熵(SGE)
print("=" * 70)
print("模型 2: SGE ~ 组别 * 阶段 * 飞行天数 (无插值)")
print("=" * 70)

model_SGE_raw = smf.mixedlm(
    "Q('静态注视熵(SGE)') ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_SGE_raw = model_SGE_raw.fit(method="powell", maxiter=500)
print(result_SGE_raw.summary())

In [ ]:
# 模型 3: 眼跳注视熵(GTE)
print("=" * 70)
print("模型 3: GTE ~ 组别 * 阶段 * 飞行天数 (无插值)")
print("=" * 70)

model_GTE_raw = smf.mixedlm(
    "Q('眼跳注视熵(GTE)') ~ 组别 * 阶段 * 飞行天数",
    df,
    groups=df["受试者"]
)
result_GTE_raw = model_GTE_raw.fit(method="powell", maxiter=500)
print(result_GTE_raw.summary())

In [ ]:
# ========================================
# 提取关键交互项对比
# ========================================

# 提取无插值结果
df_aoi_raw = extract_result(result_AOI_raw, "AOI(无插值)")
df_sge_raw = extract_result(result_SGE_raw, "SGE(无插值)")
df_gte_raw = extract_result(result_GTE_raw, "GTE(无插值)")

# 提取线性插值结果（取自 EYE/5 的 notebook 输出）
# 注：以下结果需在实际运行 EYE/5 后填入，或在此 notebook 中同时加载 EYE/5 的输出

# 关键三阶交互项列表
key_terms = [
    '组别[T.Control]:阶段[T.起飞]:飞行天数',
    '组别[T.Control]:阶段[T.转弯]:飞行天数',
    '组别[T.Control]:阶段[T.降落]:飞行天数'
]

# 构建对比表（无插值版本）
comparison_rows = []
for term in key_terms:
    for df_result, metric_name in [
        (df_aoi_raw, 'AOI转换次数'),
        (df_sge_raw, 'SGE'),
        (df_gte_raw, 'GTE')
    ]:
        if term in df_result.index:
            row = df_result.loc[term]
            comparison_rows.append({
                '指标': metric_name,
                '效应项': term,
                '无插值 β': round(row['coef'], 4),
                '无插值 p': round(row['p_value'], 4),
                '无插值 显著性': row['显著性']
            })

comparison_raw = pd.DataFrame(comparison_rows)
print("\n=== 无插值 LMM: 关键三阶交互项 ===")
print(comparison_raw.to_string(index=False))

In [ ]:
# ========================================
# 与线性插值结果的对比表
# 注：线性插值结果来自 EYE/5 的原始运行输出
# ========================================

# EYE/5 的结果（从 notebook 输出中提取的已知值）
interpolation_results = {
    ('AOI转换次数', '组别[T.Control]:阶段[T.起飞]:飞行天数'): (-0.416, 0.655),
    ('AOI转换次数', '组别[T.Control]:阶段[T.转弯]:飞行天数'): (-1.293, 0.080),
    ('AOI转换次数', '组别[T.Control]:阶段[T.降落]:飞行天数'): (-1.892, 0.044),
    ('SGE', '组别[T.Control]:阶段[T.起飞]:飞行天数'): (-0.140, 0.009),
    ('SGE', '组别[T.Control]:阶段[T.转弯]:飞行天数'): (-0.067, 0.115),
    ('SGE', '组别[T.Control]:阶段[T.降落]:飞行天数'): (-0.128, 0.017),
    ('GTE', '组别[T.Control]:阶段[T.起飞]:飞行天数'): (-0.045, 0.053),
    ('GTE', '组别[T.Control]:阶段[T.转弯]:飞行天数'): (-0.046, 0.014),
    ('GTE', '组别[T.Control]:阶段[T.降落]:飞行天数'): (-0.034, 0.150),
}

# 合并对比
full_comparison = []
for (metric, term), (beta_ip, p_ip) in interpolation_results.items():
    row_raw = comparison_raw[
        (comparison_raw['指标'] == metric) & 
        (comparison_raw['效应项'] == term)
    ]
    
    beta_raw = row_raw['无插值 β'].values[0] if len(row_raw) > 0 else np.nan
    p_raw = row_raw['无插值 p'].values[0] if len(row_raw) > 0 else np.nan
    
    direction_consistent = (np.sign(beta_ip) == np.sign(beta_raw)) if not np.isnan(beta_raw) else None
    
    full_comparison.append({
        '指标': metric,
        '效应项': term.split(':')[2] if ':' in term else term,
        '线性插值 β': beta_ip,
        '线性插值 p': p_ip,
        '无插值 β': round(beta_raw, 4) if not np.isnan(beta_raw) else 'N/A',
        '无插值 p': round(p_raw, 4) if not np.isnan(p_raw) else 'N/A',
        '方向一致': '✓' if direction_consistent else ('✗' if direction_consistent is False else 'N/A')
    })

comparison_final = pd.DataFrame(full_comparison)
print("\n=== 敏感性分析：线性插值 vs 无插值 (Pairwise Deletion) ===")
print(comparison_final.to_string(index=False))

In [ ]:
# 保存全部结果
with pd.ExcelWriter("敏感性分析_插值vs无插值.xlsx") as writer:
    comparison_final.to_excel(writer, sheet_name="关键交互项对比", index=False)
    
    # 各指标完整模型结果
    df_aoi_raw.to_excel(writer, sheet_name="AOI_无插值")
    df_sge_raw.to_excel(writer, sheet_name="SGE_无插值")
    df_gte_raw.to_excel(writer, sheet_name="GTE_无插值")

print("结果已保存至: 敏感性分析_插值vs无插值.xlsx")
print("\n=== 分析完成 ===")
print("将此对比表作为补充材料 Table S?，正文 Results 眼动段末尾加 1-2 句描述。")